In [3]:
# %% [markdown]
# # Context-Aware Chatbot - Simplified Version
# ## Works 100% in Jupyter without complex dependencies

# %% [markdown]
# ### Step 1: Install Minimal Packages

# %%
# Install only what's necessary
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install langchain streamlit ipywidgets pandas numpy
print("✅ Installation complete!")

# %% [markdown]
# ### Step 2: Simple Imports (No PyTorch needed)

# %%
import os
import json
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# For Jupyter widgets
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    IPYWIDGETS_AVAILABLE = True
    print("✅ ipywidgets available")
except ImportError:
    IPYWIDGETS_AVAILABLE = False
    print("⚠️ ipywidgets not available")

print("🎯 Setup complete!")

# %% [markdown]
# ### Step 3: Create Knowledge Base

# %%
def create_knowledge_base():
    """Create a knowledge base with text files"""
    
    # Create directory
    kb_dir = "knowledge_base"
    os.makedirs(kb_dir, exist_ok=True)
    
    # Sample documents about AI and ML
    documents = {
        "ai_intro.txt": """
        Artificial Intelligence (AI) is the simulation of human intelligence in machines 
        that are programmed to think and learn. AI can be categorized into narrow AI 
        (designed for specific tasks) and general AI (broader cognitive abilities).
        """,
        
        "ml_basics.txt": """
        Machine Learning (ML) is a subset of AI that enables systems to learn from data.
        There are three main types: supervised learning (using labeled data), 
        unsupervised learning (finding patterns in unlabeled data), and 
        reinforcement learning (using rewards and punishments).
        """,
        
        "deep_learning.txt": """
        Deep Learning is a subset of machine learning that uses neural networks with 
        multiple layers. It's particularly successful in image recognition, 
        natural language processing, and speech recognition.
        """,
        
        "nlp_basics.txt": """
        Natural Language Processing (NLP) helps computers understand human language.
        Common tasks include text classification, sentiment analysis, 
        machine translation, and question answering.
        """,
        
        "rag_explained.txt": """
        Retrieval-Augmented Generation (RAG) combines information retrieval with 
        text generation. When a query is received, the system retrieves relevant 
        documents from a knowledge base, then uses them as context for generating 
        responses.
        """,
        
        "linear_regression.txt": """
        Linear Regression is a fundamental machine learning algorithm used for predicting 
        a continuous output variable based on input features. It assumes a linear 
        relationship between inputs and output. The goal is to find the best-fitting line 
        that minimizes the sum of squared errors between predicted and actual values.
        """,
        
        "python_basics.txt": """
        Python is a popular programming language for AI and data science. It's known for 
        its simplicity and readability. Key libraries include NumPy for numerical computing, 
        Pandas for data manipulation, and Matplotlib for visualization.
        """,
        
        "chatbot_history.txt": """
        This conversation demonstrates how the chatbot maintains context. When you ask 
        follow-up questions, the bot remembers what was discussed earlier and provides 
        relevant responses based on the conversation history.
        """
    }
    
    # Write documents to files
    for filename, content in documents.items():
        filepath = os.path.join(kb_dir, filename)
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(content.strip())
    
    print(f"✅ Created {len(documents)} documents in '{kb_dir}' folder")
    return kb_dir

# Create knowledge base
kb_dir = create_knowledge_base()

# %% [markdown]
# ### Step 4: Document Store (Simple Keyword-Based)

# %%
class SimpleDocumentStore:
    """Simple document store using keyword matching"""
    
    def __init__(self):
        self.documents = []
        self.keyword_index = {}
    
    def add_documents(self, chunks):
        """Add documents to the store and build keyword index"""
        self.documents = chunks
        
        # Build keyword index
        for i, doc in enumerate(chunks):
            content = doc["content"].lower()
            # Split into words and remove common words
            words = content.split()
            for word in words:
                word = word.strip('.,!?;:()[]{}').lower()
                if len(word) > 3:  # Ignore short words
                    if word not in self.keyword_index:
                        self.keyword_index[word] = []
                    self.keyword_index[word].append(i)
        
        print(f"✅ Added {len(chunks)} documents to store")
        print(f"📚 Keyword index has {len(self.keyword_index)} unique terms")
    
    def search(self, query, k=3):
        """Search for relevant documents using keyword matching"""
        query_words = query.lower().split()
        
        # Score documents based on keyword matches
        scores = {}
        for i in range(len(self.documents)):
            scores[i] = 0
        
        for word in query_words:
            word = word.strip('.,!?;:()[]{}')
            if word in self.keyword_index:
                for doc_idx in self.keyword_index[word]:
                    scores[doc_idx] += 1
        
        # Get top k
        top_indices = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
        top_indices = [idx for idx, score in top_indices if score > 0]
        
        results = [self.documents[i] for i in top_indices]
        return results

# %% [markdown]
# ### Step 5: Load and Split Documents

# %%
def load_documents(directory_path):
    """Load documents from directory"""
    documents = []
    
    for filename in os.listdir(directory_path):
        if filename.endswith('.txt'):
            filepath = os.path.join(directory_path, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()
                documents.append({
                    "content": content,
                    "source": filename
                })
    
    print(f"📚 Loaded {len(documents)} documents")
    return documents

def split_documents(documents):
    """Split documents into chunks"""
    chunks = []
    
    for doc in documents:
        content = doc["content"]
        source = doc["source"]
        
        # Simple splitting by paragraphs
        paragraphs = content.split('\n\n')
        for i, para in enumerate(paragraphs):
            if para.strip():
                chunks.append({
                    "content": para.strip(),
                    "source": source,
                    "chunk_id": i
                })
    
    print(f"🔪 Split into {len(chunks)} chunks")
    return chunks

# Load and split
documents = load_documents(kb_dir)
chunks = split_documents(documents)

# Create document store
doc_store = SimpleDocumentStore()
doc_store.add_documents(chunks)

# Test search
test_query = "What is machine learning?"
results = doc_store.search(test_query, k=2)

print(f"\n🔍 Search results for: '{test_query}'")
print("-" * 50)
for i, result in enumerate(results, 1):
    print(f"{i}. From {result['source']}:")
    print(f"   {result['content'][:100]}...")

# %% [markdown]
# ### Step 6: Chatbot with Memory

# %%
class Chatbot:
    """Simple chatbot with memory and retrieval"""
    
    def __init__(self, doc_store):
        self.doc_store = doc_store
        self.conversation_history = []
        self.max_history = 10
        
        # Predefined responses for common topics
        self.responses = {
            "ai": """Artificial Intelligence (AI) is about creating machines that can think and learn. It includes:
• Narrow AI: Designed for specific tasks (like facial recognition)
• General AI: Hypothetical systems with broader cognitive abilities
AI encompasses machine learning, deep learning, natural language processing, and more.""",
            
            "machine learning": """Machine Learning is a subset of AI where systems learn from data. Main types:
• Supervised Learning: Learning from labeled data (e.g., classification, regression)
• Unsupervised Learning: Finding patterns in unlabeled data (e.g., clustering)
• Reinforcement Learning: Learning through rewards and punishments""",
            
            "deep learning": """Deep Learning uses neural networks with multiple layers to analyze data. It excels at:
• Image recognition and computer vision
• Natural language processing
• Speech recognition
• Game playing (like AlphaGo)""",
            
            "nlp": """Natural Language Processing (NLP) helps computers understand human language. Applications include:
• Sentiment analysis
• Machine translation
• Question answering
• Text summarization
• Chatbots and virtual assistants""",
            
            "rag": """Retrieval-Augmented Generation (RAG) combines document retrieval with text generation:
1. First, it retrieves relevant documents from a knowledge base
2. Then, it uses these documents as context for generating responses
3. This approach reduces hallucinations and ensures factual accuracy""",
            
            "linear regression": """Linear Regression is a fundamental machine learning algorithm used for predicting continuous values. It finds a linear relationship between input features and output by fitting a line that minimizes the sum of squared errors. For example, it could predict house prices based on size, number of rooms, etc.""",
            
            "python": """Python is a versatile programming language widely used in AI and data science. Key features:
• Simple and readable syntax
• Extensive libraries (NumPy, Pandas, Matplotlib, Scikit-learn)
• Great for prototyping and production
• Large community and ecosystem"""
        }
    
    def add_to_history(self, role, message):
        """Add message to conversation history"""
        self.conversation_history.append({
            "role": role,
            "message": message
        })
        # Keep only last max_history exchanges
        if len(self.conversation_history) > self.max_history:
            self.conversation_history = self.conversation_history[-self.max_history:]
    
    def get_context(self):
        """Get recent conversation context"""
        if not self.conversation_history:
            return ""
        
        context = "Previous conversation:\n"
        for exchange in self.conversation_history[-4:]:
            prefix = "User: " if exchange["role"] == "user" else "Assistant: "
            context += prefix + exchange["message"][:50] + "...\n"
        return context
    
    def get_response(self, query):
        """Generate response based on query and context"""
        
        query_lower = query.lower()
        
        # Add to history
        self.add_to_history("user", query)
        
        # Check for greetings
        if any(word in query_lower for word in ['hello', 'hi', 'hey', 'greetings']):
            response = "Hello! I'm your AI assistant. I can answer questions about AI, machine learning, deep learning, NLP, RAG, linear regression, and Python. What would you like to know?"
        
        # Check for how are you
        elif 'how are you' in query_lower:
            response = "I'm doing well, thanks for asking! I'm here to help you learn about AI and related topics."
        
        # Check for predefined topics
        elif 'ai' in query_lower and 'artificial intelligence' in query_lower:
            response = self.responses["ai"]
        elif 'machine learning' in query_lower or 'ml' in query_lower:
            response = self.responses["machine learning"]
        elif 'deep learning' in query_lower:
            response = self.responses["deep learning"]
        elif 'nlp' in query_lower or 'natural language' in query_lower:
            response = self.responses["nlp"]
        elif 'rag' in query_lower:
            response = self.responses["rag"]
        elif 'linear regression' in query_lower:
            response = self.responses["linear regression"]
        elif 'python' in query_lower:
            response = self.responses["python"]
        
        # Check for follow-up questions using context
        elif any(word in query_lower for word in ['it', 'that', 'this', 'they', 'them']):
            # Try to find the last topic discussed
            last_user_msgs = [msg for msg in self.conversation_history[-4:] if msg["role"] == "user"]
            if last_user_msgs:
                last_topic = last_user_msgs[-1]["message"].lower()
                if 'machine learning' in last_topic:
                    response = self.responses["machine learning"]
                elif 'ai' in last_topic:
                    response = self.responses["ai"]
                elif 'deep learning' in last_topic:
                    response = self.responses["deep learning"]
                else:
                    # Search in documents
                    results = self.doc_store.search(query, k=1)
                    if results:
                        response = f"Based on my knowledge: {results[0]['content'][:200]}..."
                    else:
                        response = "I'm not sure I understand. Could you please rephrase your question?"
            else:
                response = "I'm not sure what you're referring to. Could you be more specific?"
        
        # Search in documents
        else:
            results = self.doc_store.search(query, k=1)
            if results:
                response = f"Based on my knowledge: {results[0]['content'][:200]}..."
            else:
                response = "I can help you with topics about AI, Machine Learning, Deep Learning, NLP, RAG, Linear Regression, and Python. What would you like to know?"
        
        # Add response to history
        self.add_to_history("assistant", response)
        
        return response

# Initialize chatbot
chatbot = Chatbot(doc_store)

# Test the chatbot
print("\n🤖 Testing the chatbot:")
print("=" * 60)

test_queries = [
    "Hello!",
    "What is AI?",
    "Tell me about machine learning",
    "What is linear regression?",
    "Tell me about Python",
    "What is NLP?"
]

for query in test_queries:
    print(f"\n👤 User: {query}")
    response = chatbot.get_response(query)
    print(f"🤖 Assistant: {response}")
    print("-" * 60)

# %% [markdown]
# ### Step 7: Interactive Chat Interface

# %%
if IPYWIDGETS_AVAILABLE:
    def create_chat_interface():
        """Create an interactive chat interface"""
        
        # Create widgets
        chat_output = widgets.Output()
        query_input = widgets.Text(
            placeholder='Type your question here...',
            description='You:',
            layout=widgets.Layout(width='70%')
        )
        send_btn = widgets.Button(
            description='Send',
            button_style='primary',
            layout=widgets.Layout(width='15%')
        )
        clear_btn = widgets.Button(
            description='Clear',
            button_style='warning',
            layout=widgets.Layout(width='15%')
        )
        
        # Create chatbot
        interface_chatbot = Chatbot(doc_store)
        
        def send_message(b):
            query = query_input.value
            if query.strip():
                with chat_output:
                    print(f"👤 You: {query}")
                    response = interface_chatbot.get_response(query)
                    print(f"🤖 Bot: {response}\n")
                query_input.value = ''
        
        def clear_chat(b):
            chat_output.clear_output()
            interface_chatbot.conversation_history = []
            with chat_output:
                print("✨ Chat cleared! Start a new conversation.\n")
                print("💡 Try asking about:")
                print("   • What is AI?")
                print("   • Explain machine learning")
                print("   • What is linear regression?")
                print("   • Tell me about Python")
                print("   • How does deep learning work?")
        
        # Attach event handlers
        send_btn.on_click(send_message)
        clear_btn.on_click(clear_chat)
        query_input.on_submit(send_message)
        
        # Layout
        input_row = widgets.HBox([query_input, send_btn, clear_btn])
        
        # Display
        print("=" * 60)
        print("🤖 INTERACTIVE CHATBOT")
        print("Ask me about AI, ML, Deep Learning, NLP, RAG, Linear Regression, or Python")
        print("=" * 60)
        
        display(chat_output)
        display(input_row)
        
        with chat_output:
            print("Hello! I'm your AI assistant. How can I help you today?\n")
            print("💡 Try these questions:")
            print("   • What is AI?")
            print("   • Explain machine learning")
            print("   • What is linear regression?")
            print("   • Tell me about Python")
    
    # Launch interface
    create_chat_interface()
    
else:
    print("⚠️ Interactive chat not available. Using simple command line.")
    simple_chatbot = Chatbot(doc_store)
    print("\nSimple Chat (type 'quit' to exit):")
    while True:
        user_input = input("\nYou: ")
        if user_input.lower() == 'quit':
            print("Bot: Goodbye! 👋")
            break
        response = simple_chatbot.get_response(user_input)
        print(f"Bot: {response}")

# %% [markdown]
# ### Step 8: Memory Demonstration

# %%
def demonstrate_memory():
    """Show how the chatbot maintains context"""
    
    memory_bot = Chatbot(doc_store)
    
    print("\n🧠 DEMONSTRATING CONVERSATION MEMORY")
    print("=" * 60)
    
    # Conversation
    conversation = [
        "What is machine learning?",
        "What are its main types?",
        "Can you give me an example?",
        "How is it different from deep learning?"
    ]
    
    for query in conversation:
        print(f"\n👤 User: {query}")
        response = memory_bot.get_response(query)
        print(f"🤖 Bot: {response[:150]}...")
    
    print("\n📝 Conversation History:")
    print("-" * 40)
    for exchange in memory_bot.conversation_history:
        print(f"{exchange['role']}: {exchange['message'][:60]}...")
    
    return memory_bot

# Run demonstration
memory_bot = demonstrate_memory()

# %% [markdown]
# ### Step 9: Save and Load Chat

# %%
def save_chat(chatbot, filename="chat_history.json"):
    """Save chat history"""
    try:
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(chatbot.conversation_history, f, indent=2)
        print(f"✅ Chat saved to {filename}")
    except Exception as e:
        print(f"❌ Error saving: {e}")

def load_chat(chatbot, filename="chat_history.json"):
    """Load chat history"""
    try:
        if os.path.exists(filename):
            with open(filename, 'r', encoding='utf-8') as f:
                chatbot.conversation_history = json.load(f)
            print(f"✅ Chat loaded from {filename}")
            print(f"   Loaded {len(chatbot.conversation_history)} messages")
    except Exception as e:
        print(f"❌ Error loading: {e}")

# Save and load example
save_chat(memory_bot)
new_chatbot = Chatbot(doc_store)
load_chat(new_chatbot)

# %% [markdown]
# ### Step 10: Create Streamlit App

# %%
streamlit_code = '''import streamlit as st
import os
import json

# Page config
st.set_page_config(
    page_title="AI Chatbot",
    page_icon="🤖",
    layout="centered"
)

# Title
st.title("🤖 Context-Aware AI Chatbot")
st.markdown("---")

# Initialize session state
if "messages" not in st.session_state:
    st.session_state.messages = []

# Simple chatbot class
class SimpleChatbot:
    def __init__(self):
        self.responses = {
            "ai": "Artificial Intelligence (AI) simulates human intelligence in machines. It includes narrow AI (specific tasks) and general AI (broader capabilities).",
            "machine learning": "Machine Learning is a subset of AI where systems learn from data. Types: supervised, unsupervised, and reinforcement learning.",
            "ml": "Machine Learning is a subset of AI where systems learn from data. Types: supervised, unsupervised, and reinforcement learning.",
            "deep learning": "Deep Learning uses neural networks with multiple layers for tasks like image recognition and NLP.",
            "nlp": "Natural Language Processing (NLP) helps computers understand human language for translation, sentiment analysis, etc.",
            "rag": "RAG combines document retrieval with text generation for accurate, context-aware responses.",
            "linear regression": "Linear Regression predicts continuous values by finding a linear relationship between variables.",
            "python": "Python is a popular programming language for AI with libraries like NumPy, Pandas, and Scikit-learn."
        }
    
    def get_response(self, query):
        query_lower = query.lower()
        for key, response in self.responses.items():
            if key in query_lower:
                return response
        return "I can help with AI, ML, deep learning, NLP, RAG, linear regression, and Python. What would you like to know?"

# Initialize chatbot
if "chatbot" not in st.session_state:
    st.session_state.chatbot = SimpleChatbot()

# Display messages
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Chat input
if prompt := st.chat_input("Ask me about AI and Machine Learning"):
    # Add user message
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)
    
    # Generate response
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            response = st.session_state.chatbot.get_response(prompt)
            st.markdown(response)
    
    # Add assistant message
    st.session_state.messages.append({"role": "assistant", "content": response})

# Sidebar
with st.sidebar:
    st.header("About")
    st.markdown("""
    **Topics I know about:**
    - Artificial Intelligence (AI)
    - Machine Learning (ML)
    - Deep Learning
    - Natural Language Processing (NLP)
    - RAG (Retrieval-Augmented Generation)
    - Linear Regression
    - Python Programming
    
    **Sample Questions:**
    - What is AI?
    - Explain machine learning
    - What is linear regression?
    - Tell me about Python
    - How does deep learning work?
    """)
    
    if st.button("Clear Chat"):
        st.session_state.messages = []
        st.rerun()
'''

# Write Streamlit app
with open('streamlit_app.py', 'w', encoding='utf-8') as f:
    f.write(streamlit_code)

print("\n✅ Streamlit app created as 'streamlit_app.py'")
print("Run it with: streamlit run streamlit_app.py")

# %% [markdown]
# ### Step 11: Final Test

# %%
print("\n" + "=" * 60)
print("🎉 CHATBOT IS READY!")
print("=" * 60)
print(f"\n📊 Statistics:")
print(f"- Documents in knowledge base: {len(documents)}")
print(f"- Text chunks: {len(chunks)}")
print(f"- Unique keywords indexed: {len(doc_store.keyword_index)}")
print(f"- Conversation history: {len(chatbot.conversation_history)} messages")
print("\n💡 What to do next:")
print("1. Use the interactive chat widget above")
print("2. Try asking about different topics")
print("3. Test memory with follow-up questions")
print("4. Run 'streamlit run streamlit_app.py' for web interface")
print("5. Add your own .txt files to 'knowledge_base' folder")

✅ Installation complete!
✅ ipywidgets available
🎯 Setup complete!
✅ Created 8 documents in 'knowledge_base' folder
📚 Loaded 8 documents
🔪 Split into 8 chunks
✅ Added 8 documents to store
📚 Keyword index has 141 unique terms

🔍 Search results for: 'What is machine learning?'
--------------------------------------------------
1. From ml_basics.txt:
   Machine Learning (ML) is a subset of AI that enables systems to learn from data.
        There are t...
2. From deep_learning.txt:
   Deep Learning is a subset of machine learning that uses neural networks with 
        multiple layer...

🤖 Testing the chatbot:

👤 User: Hello!
🤖 Assistant: Hello! I'm your AI assistant. I can answer questions about AI, machine learning, deep learning, NLP, RAG, linear regression, and Python. What would you like to know?
------------------------------------------------------------

👤 User: What is AI?
🤖 Assistant: Based on my knowledge: This conversation demonstrates how the chatbot maintains context. When yo

Output()


🧠 DEMONSTRATING CONVERSATION MEMORY

👤 User: What is machine learning?
🤖 Bot: Hello! I'm your AI assistant. I can answer questions about AI, machine learning, deep learning, NLP, RAG, linear regression, and Python. What would yo...

👤 User: What are its main types?
🤖 Bot: Artificial Intelligence (AI) is about creating machines that can think and learn. It includes:
• Narrow AI: Designed for specific tasks (like facial r...

👤 User: Can you give me an example?
🤖 Bot: I can help you with topics about AI, Machine Learning, Deep Learning, NLP, RAG, Linear Regression, and Python. What would you like to know?...

👤 User: How is it different from deep learning?
🤖 Bot: Deep Learning uses neural networks with multiple layers to analyze data. It excels at:
• Image recognition and computer vision
• Natural language proc...

📝 Conversation History:
----------------------------------------
user: What is machine learning?...
assistant: Hello! I'm your AI assistant. I can answer questions about A...